# Cat/Dog CNN Features: What Responds, and Where?

## 1. Question and claim boundary

This is a learned-feature walkthrough: see low-, mid-, and high-level channel patterns,
connect them to parts of real cat/dog images, and examine which regions affect the
class score. It is not a two-dimensional embedding study.

The model predicts **cat=0, dog=1**, not the 37 breeds. Matched standard and PGD-trained
models let us examine the same images under a fixed digital attack. A feature picture
cannot prove semantic understanding, safety, or physical robustness. An attractive
synthetic pattern is not a real training example or a decoded memory.

The earlier breed experiment is superseded. Its numerical results are not reused.
The new two-class head needs matching new checkpoints and evidence.

In [ ]:
import json
import os
import sys
from pathlib import Path

configured_root = os.environ.get("OXFORD_PETS_NOTEBOOK_ROOT")
candidates = [Path(configured_root).resolve()] if configured_root else []
candidates.extend([Path.cwd().resolve(), *Path.cwd().resolve().parents])
ROOT = next(
    (
        candidate
        for candidate in candidates
        if (candidate / "configs/experiment.yaml").is_file()
        and (candidate / "src/notebook_support.py").is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from its project checkout")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "0"
from src.notebook_support import (
    inspect_data,
    inspect_environment,
    inspect_model,
    inventory,
    notebook_context,
    run_stage,
    show,
)

RUN_FULL_EXPERIMENT = False
# True intentionally runs preflight, both arms, attacks, feature figures, and reports.
config, RUN_FULL_EXPERIMENT, device = notebook_context(full=RUN_FULL_EXPERIMENT)
show(
    {
        "full": RUN_FULL_EXPERIMENT,
        "device": device,
        "config": str(config.path.relative_to(ROOT)),
        "config_sha256": config.sha256,
        "labels": {"cat": 0, "dog": 1},
    }
)

## 2. Editable protocol and matched comparison

The YAML is the source of truth. Both ResNet-18 arms start from identical pinned
ImageNet weights and an identical new two-class head; they use the same ordered
samples, deterministic augmentations, optimizer updates, and schedule. Only the
training inputs differ. The configured final epoch, not the best test epoch, is compared.

Changing a valid value such as epochs is allowed and creates a new run identity.
Old evidence is not silently relabeled. No automatic protocol downgrade is used.

In [ ]:
show(config.raw)

## 3. Environment and native PyTorch MPS

Select this checkout's `.venv/bin/python`: CPython 3.13.15, pinned `requirements.txt`,
PyTorch, native MPS, and tqdm. The setup uses venv/pip, not uv. Silent MPS fallback
is disabled. This cell inspects the runtime without starting training.

In [ ]:
show(inspect_environment(config, device))

## 4. Dataset and EDA

Oxford-IIIT Pet has roughly 7,349 images, 37 breeds, 12 cat breeds and 25 dog breeds.
Learning labels are species. The official test stays intact; official `trainval` is
split 80/20 **within each breed** into training/calibration. Stored breed metadata
verifies coverage; the class-balance chart counts the actual Cat/Dog targets.

Inspect actual registered counts, species imbalance, breed coverage, image sizes, and
aspect ratios. Do not infer clean accuracy from class imbalance alone. Report macro
and per-species metrics alongside overall accuracy. EDA does not alter the split.

In [ ]:
from IPython.display import Image, Markdown, display

from src.eda import generate_eda

data_summary = inspect_data(config)
eda = generate_eda(config)
show(data_summary)
show(eda["record_summary"])
for figure_key, relative_path in eda["figures"].items():
    figure_path = ROOT / relative_path
    if not figure_path.is_file():
        raise FileNotFoundError(figure_path)
    display(Markdown(f"### EDA: {figure_key.replace('_', ' ')}"))
    display(Image(filename=str(figure_path), width=1200))

## 5. Model depth and spatial features

`RGB → conv1/low network.relu → residual blocks/mid network.layer2 → high network.layer4
→ global average pool → two-class head`.

Hooks preserve spatial channel maps at low, mid, and high stages. Early responses
often involve colors/edges; later stages combine larger patterns. This is a tendency,
not proof that an individual channel is an eye, ear, or fur detector. The 512-dimensional
pooled feature is still part of the classifier, but pooled embeddings are not the
centerpiece of this walkthrough.

In [ ]:
show(inspect_model(config))

## 6. Native MPS preflight

Check data isolation, deterministic augmentation/order, identical initialization,
CPU/MPS logit parity, finite gradients, attack bounds, unchanged BatchNorm state during
attack generation, and cleanup. Memory is recorded as telemetry, not a minimum-memory
gate. Actual OOM, invalid attacks, and numerical failures preserve a failure record.

In [ ]:
show(run_stage(config, "preflight", full=RUN_FULL_EXPERIMENT, device=device))

## 7. Standard fine-tuning

Train on clean augmented images with species cross-entropy. The reference is 15 epochs,
float32 MPS, micro-batch 16 and two-step accumulation, AdamW, one-epoch warm-up followed
by cosine decay. Configuration values remain editable. Atomic epoch checkpoints and
nested tqdm progress retain the configured final checkpoint without test selection.

In [ ]:
show(run_stage(config, "train", full=RUN_FULL_EXPERIMENT, device=device, arm="standard"))

## 8. PGD-5 adversarial fine-tuning

Change only training inputs: untargeted PGD-5, `L∞ 4/255`, step `1/255`, uniform random
start, projection and clipping in raw `[0,1]` pixels before normalization. Match the
clean arm's initialization, samples, augmentations, updates, and schedule. Neither
prettier filters nor this finite training attack establishes unrestricted robustness.

In [ ]:
show(run_stage(config, "train", full=RUN_FULL_EXPERIMENT, device=device, arm="adversarial"))

## 9. Attacks, corruptions, and risk-aware evaluation

Evaluate the full official test clean and under registered noise/blur/brightness/contrast.
Use 100 fixed test images per species (200 total) for paired FGSM and PGD-20×5. Report
clean overall/macro/per-species accuracy, robust accuracy, and attack success among
clean-correct images. Finite attacks are lower-bound search, not certification.

Fit temperature and 90%-coverage confidence threshold on clean calibration only.
Compare NLL/Brier/ECE, coverage, selective risk, and tie-aware AURC after shift without
refitting. Low ECE does not mean low error or a safe classifier.

In [ ]:
show(run_stage(config, "evaluate", full=RUN_FULL_EXPERIMENT, device=device))

## 10. Learned filters, real image parts, and attribution

Channels are chosen on **clean calibration**, before inspecting fixed test anchors:
two cats and two dogs. The figure manifest records their IDs and settings.

**Read the pictures in this order:**

1. **conv1 kernels:** actual learned RGB weights. They are not maps of the pet input.
2. **Low/mid/high activation maximization:** synthetic inputs optimized to excite a
   selected channel. They show a possible high-response stimulus, not a training memory.
3. **Top real calibration patches:** high-response locations plus receptive-field boxes
   connect channels to real image regions. A late theoretical field can exceed the
   whole image; the clipped box is possible support, not equal pixel importance.
4. **Anchor activation maps and input-gradient sensitivity:** activation locates response;
   gradient locates local sensitivity. They answer different questions.
5. **Grad-CAM and occlusion:** Grad-CAM targets the **true-species logit**; occlusion
   measures the drop in the **true-species-vs-other logit margin** after masking a
   region. These are related but distinct scalar objectives: agreement does not
   exactly validate the same score. Targets stay fixed even on misclassified inputs.
   Masking can create out-of-distribution inputs; neither proves causal learning.
6. **Randomized-weight control and aggregate response charts:** explanation should depend
   on learned parameters. A changed control is necessary evidence, not proof of validity.
   Compare saved response numbers, not brightness from separately normalized panels.

These figures explain present model behavior. They cannot identify which original
training image or causal learning event created a filter. No named eye/ear/fur detector
is verified merely because a picture looks familiar.

The next cell computes this stage in full mode, or reads only already saved,
configuration-matched figures in safe mode. Missing binary evidence is never filled
with an old breed plot. Photo-containing outputs remain local and ignored.

In [ ]:
import hashlib

feature_result = run_stage(
    config,
    "represent",
    full=RUN_FULL_EXPERIMENT,
    device=device,
)
feature_manifest = ROOT / "results/generated/feature_visualizations.json"
if isinstance(feature_result, dict) and feature_result.get("results_available") is False:
    if feature_manifest.is_file():
        feature_result = json.loads(feature_manifest.read_text(encoding="utf-8"))
    else:
        show(feature_result)

if isinstance(feature_result, dict) and "figures" in feature_result:
    if feature_result.get("config_sha256") != config.sha256:
        raise RuntimeError("Saved feature figures belong to a different configuration")
    figures = feature_result["figures"]
    if not isinstance(figures, list):
        raise TypeError("Feature manifest figures must be a list")
    display(
        Markdown(
            "### Saved learned-feature walkthrough\n\n"
            f"Evidence completed: {feature_result.get('created_at', 'not recorded')}. "
            "Pet-photo panels are local-only; source guide stays output-free."
        )
    )
    for figure in figures:
        if not isinstance(figure, dict) or not isinstance(figure.get("path"), str):
            raise TypeError("Invalid feature figure record")
        figure_path = (ROOT / figure["path"]).resolve()
        if not figure_path.is_relative_to(ROOT) or not figure_path.is_file():
            show(
                {
                    "status": "stale feature evidence",
                    "figure": figure["path"],
                    "reason": "figure is missing or outside the project",
                }
            )
            continue
        expected_sha256 = figure.get("sha256")
        try:
            current_sha256 = hashlib.sha256(figure_path.read_bytes()).hexdigest()
        except OSError as error:
            show(
                {
                    "status": "stale feature evidence",
                    "figure": figure["path"],
                    "reason": f"figure cannot be read: {error}",
                }
            )
            continue
        if not isinstance(expected_sha256, str) or current_sha256 != expected_sha256:
            show(
                {
                    "status": "stale feature evidence",
                    "figure": figure["path"],
                    "reason": "recorded figure hash is missing or mismatched",
                }
            )
            continue
        sharing = (
            "aggregate/synthetic export candidate" if figure.get("shareable") else "local-only"
        )
        display(
            Markdown(
                f"### {figure.get('arm', '')}: {figure.get('kind', 'feature view')}\n\n"
                f"{figure.get('caption', '')}\n\nSharing: **{sharing}**."
            )
        )
        display(Image(filename=str(figure_path), width=1200))
    show(feature_result.get("limitations", []))

## 11. Reports, interpretation, and local artifact inventory

Reports must use complete matching binary evidence. Interpret actual channel responses
and class-score changes separately from possible semantic stories. One model pair,
one split/seed family, ImageNet pretraining, selected anchors, explanation-method limits,
and finite attacks constrain conclusions.

A completed picture-rich local result companion is saved under `reports/generated/`
and can be read without rerunning training. It contains real photographs and remains
ignored. Only reviewed aggregate charts/synthetic features may be exported publicly.
Do not save this executed guide over the tracked output-free source.

In [ ]:
report_result = run_stage(
    config,
    "report",
    full=RUN_FULL_EXPERIMENT,
    device=device,
)
show(report_result)
show(inventory(config))

## 12. Reproduction and references

Full run: deliberately set `RUN_FULL_EXPERIMENT = True`, rerun setup, then run cells
in order. CLI: `.venv/bin/python src/cli.py reproduce --config configs/experiment.yaml
--device mps`. Existing checkpoint/evidence reuse requires matching provenance.

[Lee et al. 2009](https://ai.stanford.edu/~ang/papers/icml09-ConvolutionalDeepBeliefNetworks.pdf)
inspires the low-to-high illustration; this discriminative ResNet does **not** reproduce
their generative convolutional deep belief network. Original method references:
[Zeiler/Fergus](https://arxiv.org/abs/1311.2901),
[Grad-CAM](https://arxiv.org/abs/1610.02391),
[Adebayo sanity checks](https://arxiv.org/abs/1810.03292), and
[Distill Feature Visualization](https://distill.pub/2017/feature-visualization/).

### Current public-safe feature gallery — no execution needed

These are saved, configuration-matched synthetic/kernel/aggregate figures, with
verified asset hashes. They are linked from this checkout rather than embedded as
code outputs. Synthetic tiles are optimized channel stimuli, not reconstructed pet
photographs or proof of named eye/ear detectors. Real input patches and attribution
overlays remain in the ignored local result companion. Separately normalized tiles
cannot establish absolute response strength or robustness.

A gray zero-response tile can be an unsuccessful single-start stimulus
optimization, not a dead channel or proof that nothing was learned. Check the
recorded response gains and real calibration responses in `docs/results.json`
and the visible failure notes in `docs/results.md`; failed trials are retained.

### experiment: species response

![experiment species response](../docs/assets/experiment-species_response-experiment-workflow.png)

Measured-layer architecture and distinct probe questions: synthetic channel preferences, spatial responses/local sensitivity, and class attribution. Intermediate layer1/layer3 are included in the forward path, although not selected for image grids.

### standard: activation maximization

![standard activation maximization](../docs/assets/standard-activation_maximization-standard-synthetic-atlas.png)

Layer-wise optimized channel preferences. Every tile is independently optimized from private seeded noise; visual appearance is not evidence of a named detector.

### adversarial: activation maximization

![adversarial activation maximization](../docs/assets/adversarial-activation_maximization-adversarial-synthetic-atlas.png)

Layer-wise optimized channel preferences. Every tile is independently optimized from private seeded noise; visual appearance is not evidence of a named detector. WARNING: this arm predicts only dog on all 3669 clean test images (macro accuracy 50%). Nominal attack survival must not be presented as useful robust cat/dog recognition. This is a classifier prediction failure, not proof that all hidden features are constant.

### experiment: species response

![experiment species response](../docs/assets/experiment-species_response-experiment-accuracy-context.png)

Saved classification evidence: clean official-test accuracy is separate from matched-subset clean/FGSM/PGD accuracy. Finite attacks do not certify robustness, and these values do not validate a named feature. WARNING: this arm predicts only dog on all 3669 clean test images (macro accuracy 50%). Nominal attack survival must not be presented as useful robust cat/dog recognition. This is a classifier prediction failure, not proof that all hidden features are constant.

### standard: kernels

![standard kernels](../docs/assets/standard-kernels-standard-kernels.png)

Actual first-layer weights, initialized weights, and their differences. Deeper channel filters have many input channels and are instead probed with optimization and real examples.

### adversarial: kernels

![adversarial kernels](../docs/assets/adversarial-kernels-adversarial-kernels.png)

Actual first-layer weights, initialized weights, and their differences. Deeper channel filters have many input channels and are instead probed with optimization and real examples. WARNING: this arm predicts only dog on all 3669 clean test images (macro accuracy 50%). Nominal attack survival must not be presented as useful robust cat/dog recognition. This is a classifier prediction failure, not proof that all hidden features are constant.